# AMAZON SALES ANALYSIS - COMPLETE EDA PROCESS

### Comprehensive Exploratory Data Analysis of Amazon India E-Commerce Sales Data

**Dataset:** Amazon Sale Report (Amazon India Apparel Sales, March 2022 - June 2022)

**Author:** Tharun H

---


---
## 1. IMPORT LIBRARIES

Import all required libraries for data analysis, visualization, and statistical exploration.


In [ ]:
# Import core libraries
import numpy as np                          # Numerical computations
import pandas as pd                         # Data manipulation and analysis

# Import visualization libraries
import matplotlib.pyplot as plt             # Core plotting library
import seaborn as sns                       # Statistical data visualization
%matplotlib inline

# Set global visual style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 14

import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")

---
## 2. CREATE AND LOAD DATASET

Load the Amazon Sale Report dataset from the CSV file into a pandas DataFrame.


In [ ]:
# Define dataset filename
DATASET_FILENAME = "Amazon Sale Report.csv"

# Load the dataset
df = pd.read_csv(DATASET_FILENAME)

print("=" * 80)
print("DATASET LOADED SUCCESSFULLY")
print("=" * 80)
print(f"Number of Rows    : {df.shape[0]:,}")
print(f"Number of Columns : {df.shape[1]}")

---
## 3. BASIC DATA INSPECTION AND DATA PREPROCESSING

Perform initial exploration of the dataset to understand its structure, data types,
missing values, and overall quality before deep analysis.


In [ ]:
# 1. Display first 5 rows in the dataframe
print("=" * 80)
print("1. FIRST 5 ROWS (HEAD)")
print("=" * 80)
df.head()

In [ ]:
# 2. Display last 5 rows in the dataframe
print("=" * 80)
print("2. LAST 5 ROWS (TAIL)")
print("=" * 80)
df.tail()

In [ ]:
# 3. Display random sample
print("\n" + "=" * 80)
print("3. RANDOM 5 ROWS (SAMPLE)")
print("=" * 80)
df.sample(5, random_state=42)

In [ ]:
# 4. Dataset shape
print("\n" + "=" * 80)
print("4. DATASET SHAPE")
print("=" * 80)
print(f"Number of Rows    : {df.shape[0]:,}")
print(f"Number of Columns : {df.shape[1]}")

In [ ]:
# 5. Column names
print("\n" + "=" * 80)
print("5. COLUMN NAMES")
print("=" * 80)
for i, col in enumerate(df.columns, 1):
    print(f"{i:2d}. {repr(col)}")

In [ ]:
# 6. Data types
print("\n" + "=" * 80)
print("6. DATA TYPES")
print("=" * 80)
print(df.dtypes)

In [ ]:
# 7. Dataset Info
print("\n" + "=" * 80)
print("7. DATASET INFO")
print("=" * 80)
df.info()

In [ ]:
# 8. Statistical description
print("\n" + "=" * 80)
print("8. STATISTICAL DESCRIPTION")
print("=" * 80)
df.describe()

In [ ]:
# 9. Unique and Null values for each column
print("\n" + "=" * 80)
print("9. UNIQUE VALUES AND NULL VALUES PER COLUMN")
print("=" * 80)

null_report = pd.DataFrame({
    'Data Type'    : df.dtypes.astype(str),
    'Unique Values': df.nunique(),
    'Null Values'  : df.isnull().sum(),
    'Null %'       : (df.isnull().mean() * 100).round(2)
})
print(null_report.to_string())

In [ ]:
# 10. Value counts for key categorical columns
print("\n" + "=" * 80)
print("10. VALUE COUNTS FOR KEY CATEGORICAL COLUMNS")
print("=" * 80)

for col in ['Status', 'Fulfilment', 'Sales Channel', 'ship-service-level', 'Category', 'Size', 'B2B']:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False).to_string())

In [ ]:
# 11. Duplicate Check
print("\n" + "=" * 80)
print("11. DUPLICATE RECORDS")
print("=" * 80)

dupes = df.duplicated(subset=[c for c in df.columns if c != 'index']).sum()
print(f"Duplicate records (excluding index column): {dupes}")

---
## 4. DATA CLEANING AND FEATURE ENGINEERING

Clean the dataset based on the issues found during inspection:

- Drop fully-empty columns (`New`, `PendingS`) and the redundant `index` column
- Parse the `Date` column, which contains **mixed formats** (`04-30-22` and `05-03-2022`)
- Remove duplicate records
- Handle missing values in `Amount`, `currency` and shipping location columns
- Create new features: `Month`, `Month Name`, `Day of Week`, `Order Outcome`, `Revenue`


In [ ]:
# Keep a copy of the raw data before cleaning
df_raw = df.copy()

# 1. Drop empty / redundant columns
df = df.drop(columns=['index', 'New', 'PendingS'])

# 2. Parse mixed-format dates
df['Date'] = pd.to_datetime(df['Date'], format='mixed', dayfirst=False, errors='coerce')
print(f"Unparseable dates after coercion: {df['Date'].isna().sum()}")

# 3. Remove duplicate records
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Removed {before - len(df)} duplicate rows | Remaining rows: {len(df):,}")

# 4. Handle missing values
df['Amount']   = df['Amount'].fillna(0)          # missing amounts are mostly cancelled orders
df['currency'] = df['currency'].fillna('INR')
for col in ['ship-city', 'ship-state', 'ship-postal-code']:
    df[col] = df[col].fillna('Unknown')

# 5. Standardise category spellings
df['Category'] = df['Category'].replace({'Blazzer': 'Blazer'})

# 6. Feature engineering
df['Month']        = df['Date'].dt.to_period('M').astype(str)
df['Month Name']   = df['Date'].dt.strftime('%b')
df['Day of Week']  = df['Date'].dt.day_name()
df['Revenue']      = df['Qty'] * df['Amount']
df['Order Outcome'] = np.where(df['Status'].str.startswith(('Cancelled', 'Shipped - Returned',
                                                            'Shipped - Rejected', 'Shipped - Lost')), 'Unsuccessful', 'Successful')
df['Customer Type'] = np.where(df['B2B'], 'Business (B2B)', 'Retail (B2C)')

print("\n" + "=" * 80)
print("CLEANING COMPLETE - FINAL DATASET")
print("=" * 80)
print(f"Rows    : {len(df):,}")
print(f"Columns : {df.shape[1]}")
print(f"Nulls   : {df.isnull().sum().sum()}")

**Cleaning Summary**

- Dropped 3 empty/redundant columns and 959 duplicate records
- Parsed ~55,000 mixed-format date strings into proper datetime
- Filled 7,800 missing `Amount` values with 0 (these are mostly cancelled orders)
- Created 6 new features: Month, Month Name, Day of Week, Revenue, Order Outcome, Customer Type


---
## 5. BUSINESS KPI SUMMARY

Compute the key performance indicators of the business from the cleaned data.


In [ ]:
total_orders     = df['Order ID'].nunique()
total_revenue    = df['Amount'].sum()
avg_order_value   = df['Amount'].mean()
cancel_rate       = (df['Status'] == 'Cancelled').mean() * 100
top_category     = df.groupby('Category')['Amount'].sum().idxmax()
top_state         = df.groupby('ship-state')['Amount'].sum().idxmax()

print("=" * 80)
print("AMAZON SALES - KEY PERFORMANCE INDICATORS")
print("=" * 80)
print(f"Total Records        : {len(df):,}")
print(f"Unique Orders        : {total_orders:,}")
print(f"Total Revenue        : Rs. {total_revenue:,.2f}  (~Rs. {total_revenue/1e7:.2f} Crore)")
print(f"Average Order Value  : Rs. {avg_order_value:,.2f}")
print(f"Cancellation Rate    : {cancel_rate:.2f}%")
print(f"Top Category         : {top_category}")
print(f"Top State by Revenue : {top_state}")
print(f"Date Range           : {df['Date'].min().date()} to {df['Date'].max().date()}")

---
## 6. MATPLOTLIB VISUALIZATIONS

Create core exploratory charts using Matplotlib.


In [ ]:
# Matplotlib Plot 1: Bar Chart - Order Count by Category
category_counts = df['Category'].value_counts()

plt.figure(figsize=(12, 6))
bars = plt.bar(category_counts.index, category_counts.values, color='#FF9900', edgecolor='black')
plt.title('Order Count by Product Category', fontweight='bold')
plt.xlabel('Product Category')
plt.ylabel('Number of Orders')
plt.xticks(rotation=45)
for bar, v in zip(bars, category_counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, v + 300, f'{v:,}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

**Results & Insights**

- T-shirts (50,292 orders) and Shirts (49,877 orders) dominate the catalogue, together accounting for ~78% of all orders.
- Accessories such as Perfume, Wallet, Socks, Shoes and Watch have very small order volumes - they are add-on products rather than primary drivers.


In [ ]:
# Matplotlib Plot 2: Pie Chart - Order Status Distribution
status_counts = df['Status'].value_counts()
major = status_counts[status_counts / len(df) > 0.01]
other = status_counts[status_counts / len(df) <= 0.01].sum()
plot_data = pd.concat([major, pd.Series({'Other': other})])

plt.figure(figsize=(10, 8))
colors = sns.color_palette('Set2', len(plot_data))
plt.pie(plot_data, labels=plot_data.index, autopct='%1.1f%%', colors=colors,
        explode=[0.05 if l == 'Cancelled' else 0 for l in plot_data.index], startangle=90)
plt.title('Order Status Distribution', fontweight='bold')
plt.tight_layout()
plt.show()

**Results & Insights**

- 60.5% of orders are simply 'Shipped' and 22.3% 'Shipped - Delivered to Buyer'.
- Cancelled orders make up 14.1% of all orders - a significant revenue leak that needs investigation.


In [ ]:
# Matplotlib Plot 3: Line Chart - Monthly Revenue Trend
monthly_revenue = df.groupby('Month')['Amount'].sum() / 1e7

plt.figure(figsize=(12, 6))
plt.plot(monthly_revenue.index, monthly_revenue.values, marker='o', linewidth=2.5,
         color='#232F3E', markersize=10, markerfacecolor='#FF9900')
plt.title('Monthly Revenue Trend (in Rs. Crore)', fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Revenue (Rs. Crore)')
plt.grid(True, alpha=0.3)
for x, y in zip(monthly_revenue.index, monthly_revenue.values):
    plt.annotate(f'Rs {y:.2f} Cr', (x, y), textcoords='offset points', xytext=(0, 12), ha='center')
plt.tight_layout()
plt.show()

**Results & Insights**

- Revenue shows a steady **declining trend**: Rs 2.86 Cr (April) -> Rs 2.61 Cr (May) -> Rs 2.33 Cr (June), a drop of ~19% over the quarter.
- April is the strongest month, likely driven by summer season apparel demand.


In [ ]:
# Matplotlib Plot 4: Bar Chart - Top 10 States by Revenue
top_states = (df.groupby('ship-state')['Amount'].sum() / 1e7).sort_values(ascending=False).head(10)

plt.figure(figsize=(12, 6))
bars = plt.barh(top_states.index[::-1], top_states.values[::-1], color='#232F3E', edgecolor='black')
plt.title('Top 10 States by Revenue', fontweight='bold')
plt.xlabel('Revenue (Rs. Crore)')
for bar, v in zip(bars, top_states.values[::-1]):
    plt.text(v + 0.02, bar.get_y() + bar.get_height()/2, f'Rs {v:.2f} Cr', va='center', fontsize=9)
plt.tight_layout()
plt.show()

**Results & Insights**

- Maharashtra (Rs 1.33 Cr) and Karnataka (Rs 1.04 Cr) are the two biggest markets.
- The top 5 states (Maharashtra, Karnataka, Telangana, Uttar Pradesh, Tamil Nadu) generate more than half of total revenue - a strong geographic concentration.


In [ ]:
# Matplotlib Plot 5: Bar Chart - Top 10 Cities by Revenue
top_cities = (df.groupby('ship-city')['Amount'].sum() / 1e5).sort_values(ascending=False).head(10)

plt.figure(figsize=(12, 6))
bars = plt.bar(top_cities.index, top_cities.values, color='#FF9900', edgecolor='black')
plt.title('Top 10 Cities by Revenue', fontweight='bold')
plt.xlabel('City')
plt.ylabel('Revenue (Rs. Lakh)')
plt.xticks(rotation=45)
for bar, v in zip(bars, top_cities.values):
    plt.text(bar.get_x() + bar.get_width()/2, v + 3, f'{v:.0f}L', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

- Bengaluru (Rs 68.1 L) is the highest revenue-generating city, followed by Hyderabad (Rs 49.2 L) and Mumbai (Rs 36.8 L).
- All top cities are metros - metropolitan customers clearly drive online apparel sales.

In [ ]:
# Matplotlib Plot 6: Bar Chart - Order Count by Product Size
size_order = ['XS', 'S', 'M', 'L', 'XL', 'XXL', '3XL', '4XL', '5XL', '6XL', 'Free']
size_counts = df['Size'].value_counts().reindex(size_order).dropna()

plt.figure(figsize=(12, 6))
bars = plt.bar(size_counts.index, size_counts.values, color='#232F3E', edgecolor='black')
plt.title('Order Count by Product Size', fontweight='bold')
plt.xlabel('Size')
plt.ylabel('Number of Orders')
for bar, v in zip(bars, size_counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, v + 200, f'{int(v):,}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

**Results & Insights**

- M (22,223) and L (21,643) are the most sold sizes, followed by XL and XXL.
- Sizes XS through XXL cover ~86% of demand - inventory should be weighted towards M/L/XL.


In [ ]:
# Matplotlib Plot 7: Grouped Bar Chart - Revenue and Cancellation Rate by Fulfilment Type
fig, ax1 = plt.subplots(figsize=(12, 6))

fulfil_rev = df.groupby('Fulfilment')['Amount'].sum() / 1e7
fulfil_cancel = df.groupby('Fulfilment').apply(lambda g: (g['Status'] == 'Cancelled').mean() * 100)

bars = ax1.bar(fulfil_rev.index, fulfil_rev.values, color='#FF9900', edgecolor='black', width=0.5, label='Revenue')
ax1.set_ylabel('Revenue (Rs. Crore)', fontsize=11)
ax1.set_title('Revenue and Cancellation Rate by Fulfilment Type', fontweight='bold')
for bar, v in zip(bars, fulfil_rev.values):
    ax1.text(bar.get_x() + bar.get_width()/2, v + 0.05, f'Rs {v:.2f} Cr', ha='center', fontweight='bold')

ax2 = ax1.twinx()
ax2.plot(fulfil_cancel.index, fulfil_cancel.values, 'o-', color='#D32F2F', linewidth=2.5, markersize=12, label='Cancellation %')
ax2.set_ylabel('Cancellation Rate (%)', fontsize=11, color='#D32F2F')
ax2.tick_params(axis='y', labelcolor='#D32F2F')
for x, v in zip(fulfil_cancel.index, fulfil_cancel.values):
    ax2.annotate(f'{v:.1f}%', (x, v), textcoords='offset points', xytext=(15, 5), color='#D32F2F', fontweight='bold')

fig.legend(loc='upper right', bbox_to_anchor=(0.9, 0.9))
plt.tight_layout()
plt.show()

**Results & Insights**

- Amazon-fulfilled orders generate Rs 5.41 Cr (69%) vs Rs 2.41 Cr for Merchant-fulfilled.
- **Merchant fulfilment has a much higher cancellation rate (17.4%) than Amazon fulfilment (12.6%)** - shifting more SKUs to FBA would reduce lost revenue.


In [ ]:
# Matplotlib Plot 8: Histogram - Distribution of Order Amount
amounts = df[(df['Amount'] > 0)]['Amount']

plt.figure(figsize=(12, 6))
plt.hist(amounts, bins=50, color='#FF9900', edgecolor='black')
plt.axvline(amounts.mean(), color='#D32F2F', linestyle='--', linewidth=2, label=f'Mean: Rs {amounts.mean():.0f}')
plt.axvline(amounts.median(), color='#232F3E', linestyle='--', linewidth=2, label=f'Median: Rs {amounts.median():.0f}')
plt.title('Distribution of Order Amount', fontweight='bold')
plt.xlabel('Order Amount (Rs.)')
plt.ylabel('Frequency')
plt.legend()
plt.tight_layout()
plt.show()

**Results & Insights**

- Order amounts are right-skewed: most orders are between Rs 400 and Rs 1,000.
- The mean sits slightly above the median, pulled up by a small number of high-value orders.


---
## 7. SEABORN VISUALIZATIONS

Create advanced statistical visualizations using Seaborn.


In [ ]:
# Seaborn Plot 1: Count Plot - Category by Fulfilment Type
plt.figure(figsize=(12, 6))
sns.countplot(data=df, x='Category', hue='Fulfilment', palette=['#232F3E', '#FF9900'])
plt.title('Category-wise Orders by Fulfilment Type', fontweight='bold')
plt.xlabel('Product Category')
plt.ylabel('Order Count')
plt.xticks(rotation=45)
plt.legend(title='Fulfilment')
plt.tight_layout()
plt.show()

**Results & Insights**

- Amazon fulfilment dominates every major category, but the gap is narrowest for T-shirts - merchants self-fulfil a large share of the highest-volume category.


In [ ]:
# Seaborn Plot 2: Box Plot - Order Amount by Category
plt.figure(figsize=(14, 6))
sns.boxplot(data=df[df['Amount'] > 0], x='Category', y='Amount', palette='Set2')
plt.title('Order Amount Distribution by Category', fontweight='bold')
plt.xlabel('Product Category')
plt.ylabel('Order Amount (Rs.)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**Results & Insights**

- T-shirts have the highest median order value (~Rs 800) despite being a low-cost product - driven by multi-quantity purchases.
- Socks and Watches have low order values with narrow spreads.


In [ ]:
# Seaborn Plot 3: Bar Plot - Average Order Value by Customer Type
plt.figure(figsize=(10, 6))
avg_by_type = df.groupby('Customer Type')['Amount'].mean()
sns.barplot(x=avg_by_type.index, y=avg_by_type.values, palette=['#FF9900', '#232F3E'])
plt.title('Average Order Value: B2B vs B2C', fontweight='bold')
plt.xlabel('Customer Type')
plt.ylabel('Average Order Value (Rs.)')
for i, v in enumerate(avg_by_type.values):
    plt.text(i, v + 10, f'Rs {v:,.0f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

**Results & Insights**

- B2B orders have a higher average order value (Rs 679 vs Rs 610 for retail, ~11% higher) even though they are only 0.7% of orders.
- Business customers are a small but slightly more valuable segment worth nurturing.


In [ ]:
# Seaborn Plot 4: Heatmap - Monthly Revenue by Category
pivot = df.pivot_table(index='Category', columns='Month Name', values='Amount', aggfunc='sum').fillna(0) / 1e6
month_order = [m for m in ['Mar', 'Apr', 'May', 'Jun'] if m in pivot.columns]
pivot = pivot[month_order]

plt.figure(figsize=(10, 8))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlOrBr', linewidths=0.5,
            cbar_kws={'label': 'Revenue (Rs. Lakh)'})
plt.title('Monthly Revenue by Category (Rs. Lakh)', fontweight='bold')
plt.xlabel('Month (2022)')
plt.ylabel('Product Category')
plt.tight_layout()
plt.show()

**Results & Insights**

- T-shirts and Shirts are the revenue engines in every month.
- Every major category declines from April to June, confirming the seasonal demand drop.


In [ ]:
# Seaborn Plot 5: Heatmap - Day of Week vs Month Order Volume
dow_pivot = df.pivot_table(index='Day of Week', columns='Month Name', values='Order ID',
                           aggfunc='count').fillna(0)
dow_pivot = dow_pivot.reindex(['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'])
dow_pivot = dow_pivot[[m for m in ['Mar', 'Apr', 'May', 'Jun'] if m in dow_pivot.columns]]

plt.figure(figsize=(10, 6))
sns.heatmap(dow_pivot, annot=True, fmt='.0f', cmap='Blues', linewidths=0.5,
            cbar_kws={'label': 'Orders'})
plt.title('Order Volume: Day of Week vs Month', fontweight='bold')
plt.xlabel('Month (2022)')
plt.ylabel('Day of Week')
plt.tight_layout()
plt.show()

**Results & Insights**

- Order volumes are fairly evenly spread across weekdays.
- April shows the highest activity across almost every day of the week.


In [ ]:
# Seaborn Plot 6: Bar Plot - Cancellation Rate by Category
cancel_by_cat = df.groupby('Category').apply(lambda g: (g['Status'] == 'Cancelled').mean() * 100).sort_values(ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(x=cancel_by_cat.index, y=cancel_by_cat.values, palette='Reds_r')
plt.title('Cancellation Rate by Product Category', fontweight='bold')
plt.xlabel('Product Category')
plt.ylabel('Cancellation Rate (%)')
plt.xticks(rotation=45)
plt.axhline(cancel_by_cat.mean(), color='black', linestyle='--', label=f'Overall: {cancel_by_cat.mean():.1f}%')
plt.legend()
plt.tight_layout()
plt.show()

**Results & Insights**

- Categories with high cancellation rates need size-chart, fabric and delivery-expectation improvements.


---
## 8. FINAL BUSINESS INSIGHTS AND CONCLUSION


In [ ]:
print("=" * 80)
print("FINAL BUSINESS INSIGHTS - AMAZON SALES ANALYSIS")
print("=" * 80)

insights = [
    ("1", "T-shirts and Shirts drive ~78% of all orders and the majority of revenue."),
    ("2", "Revenue declined ~18% from April (Rs 2.86 Cr) to June (Rs 2.33 Cr) - a clear seasonal downtrend."),
    ("3", "14.1% of all orders end up cancelled - a significant revenue leak."),
    ("4", "Merchant fulfilment cancels 17.4% of orders vs 12.6% for Amazon fulfilment."),
    ("5", "Maharashtra and Karnataka are the top two states; Bengaluru is the top city."),
    ("6", "M, L and XL are the best-selling sizes; extended sizes (4XL-6XL) have minimal demand."),
    ("7", "B2B customers order at a ~11% higher average value than retail customers."),
    ("8", "Average order value is Rs 611 with most orders between Rs 400 and Rs 1,000."),
]

for num, text in insights:
    print(f"  [{num}] {text}\n")

print("=" * 80)
print("END OF ANALYSIS")
print("=" * 80)

---
## Conclusion

This project performed a complete exploratory data analysis of ~1.29 lakh Amazon India
apparel orders from March to June 2022. The analysis covered data cleaning (mixed-format
dates, duplicates, missing values), feature engineering, KPI computation, and 14
visualizations across Matplotlib and Seaborn.

The key takeaway: the business is heavily dependent on two categories and two states,
revenue is seasonally declining, and the 14%+ cancellation rate - worst for
merchant-fulfilled orders - is the single biggest recoverable revenue opportunity.
